# Retrieval Augmented Generation (RAG) with LangChain
*Using IBM Granite Models*

## In this notebook

This notebook demonstrates Retrieval Augmented Generation (RAG) — an architectural pattern that improves the accuracy and relevance of language model responses by retrieving factual information from a knowledge base and injecting it into the model's context before generation.

RAG is commonly used for:

- **Customer service** — answering product/service questions using facts pulled from documentation.
- **Domain knowledge** — exploring a specialized field (e.g., finance, law, medicine) using facts from papers or articles.
- **News & current events** — chatting about recent events by retrieving relevant news articles.

At a high level, RAG works in three steps:

1. **Index** — split the knowledge base into passages, embed them, and store the embeddings in a vector database.
2. **Retrieve** — given a user query, embed the query and retrieve the most semantically similar passages.
3. **Generate** — feed the retrieved passages, along with the original query, into a large language model to produce a grounded answer.

## Setting up the environment

Install dependencies.

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    transformers \
    langchain_classic \
    langchain_community \
    langchain_text_splitters \
    langchain_huggingface sentence_transformers \
    langchain_chroma chromadb \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    wget
! echo "::endgroup::"

## Selecting System Components

### Choose your Embeddings Model

Specify the model used to generate embedding vectors from text.

To use a model from a provider other than Hugging Face, swap this cell out for one from the [Embeddings Model recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Embeddings_Models.ipynb).

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

embeddings_model_path = "ibm-granite/granite-embedding-small-english-r2"
embeddings_model = HuggingFaceEmbeddings(
    model_name=embeddings_model_path,
)
embeddings_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_path)

### Choose your Vector Database

Specify the database used to store and retrieve embedding vectors.

To connect to a vector database other than ChromaDB, swap this cell out for one from the [Vector Store recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Vector_Stores.ipynb).

In [ ]:
from langchain_chroma import Chroma

vector_db = Chroma(embedding_function=embeddings_model)

### Choose your LLM

The LLM answers the question, using the retrieved text as context.

Select a Granite language model from the [`ibm-granite`](https://replicate.com/ibm-granite) org on Replicate. Here we use the Replicate LangChain client to connect to the model.

To connect to a model on a provider other than Replicate, swap this cell out for one from the [LLM component recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_LLMs.ipynb).

In [ ]:
from langchain_replicate import ChatReplicate
from ibm_granite_community.notebook_utils import get_env_var

model_path = "ibm-granite/granite-4.1-8b"
model = ChatReplicate(
    model=model_path,
    replicate_api_token=get_env_var('REPLICATE_API_TOKEN'),
)

## Building the Vector Database

In this example, we take the State of the Union speech text, split it into chunks, derive embedding vectors using the embedding model, and load it into the vector database for querying.

### Download the document

Here we use President Biden's State of the Union address from March 1, 2022.

In [ ]:
import os
import wget

filename = 'state_of_the_union.txt'
url = 'https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/data/foundation_models/state_of_the_union.txt'

if not os.path.isfile(filename):
  wget.download(url, out=filename)

### Split the document into chunks

Split the document into text segments that fit within the model's context window.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader(filename)
documents = loader.load()
text_splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=embeddings_tokenizer,
    chunk_size=embeddings_tokenizer.max_len_single_sentence,
    chunk_overlap=0,
)
texts = text_splitter.split_documents(documents)
doc_id = 0
for text in texts:
    text.metadata["doc_id"] = (doc_id:=doc_id+1)
print(f"{len(texts)} text document chunks created")

### Populate the vector database

> **Note:** populating the vector database may take over a minute depending on your embedding model and service.

In [ ]:
ids = vector_db.add_documents(texts)
print(f"{len(ids)} documents added to the vector database")

## Querying the Vector Database

### Conduct a similarity search

Search the database for similar documents by proximity of the embedded vector in vector space.

In [ ]:
query = "What did the president say about Ketanji Brown Jackson?"
docs = vector_db.similarity_search(query)
print(f"{len(docs)} documents returned")
for doc in docs:
    print(doc)
    print("=" * 80)  # Separator for clarity

## Answering Questions

### Automate the RAG pipeline

Build a RAG chain with the model and the document retriever.

First, we create the prompt for Granite to perform the RAG query, using the Granite chat template with placeholder values that the LangChain RAG pipeline will replace.

Then we assemble the RAG pipeline using that prompt template.

In [ ]:
from ibm_granite_community.langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

# Create a Granite prompt for question-answering with the retrieved context
prompt_template = ChatPromptTemplate.from_template("{input}")

# Assemble the retrieval-augmented generation chain
combine_docs_chain = create_stuff_documents_chain(
    llm=model,
    prompt=prompt_template,
)
rag_chain = create_retrieval_chain(
    retriever=vector_db.as_retriever(),
    combine_docs_chain=combine_docs_chain,
)

### Generate a retrieval-augmented response to a question

Use the RAG chain to process a question. The document chunks relevant to that question are retrieved automatically and used as context for the answer.

In [ ]:
output = rag_chain.invoke({"input": query})

print(output['answer'])